# Query Decomposition: Breaking Complex Questions into Searchable Parts

A question can ask for several facts at once. Searching the whole question as one string may cause retrieval to focus on only one part, especially when the parts use different vocabulary or refer to separate documents. **Query decomposition** turns a complex question into smaller, answerable sub-questions, retrieves evidence for each, and checks that every part is covered.

This notebook builds a small, reproducible pipeline with a lexical teaching retriever and a product-support example. It distinguishes decomposition from paraphrase-based multi-query retrieval, tracks which evidence supports which sub-question, explores merging and coverage, and includes two SVG charts. The code uses only the Python standard library.

## Learning goals

By the end, you should be able to:

- Recognize when a question contains multiple independently answerable information needs.
- Decompose a compound question into concise, faithful sub-questions.
- Retrieve and judge evidence separately for each sub-question.
- Measure sub-question coverage, not only overall document relevance.
- Merge results without losing which evidence supports which part.
- Identify over-decomposition, missing dependencies, query drift, and unsupported answers.
- Interpret charts that show evidence coverage and the evidence-to-sub-question relationship.

## 1. What is query decomposition?

Query decomposition converts one complex question into a set of smaller searches. Each sub-question represents a distinct fact or evidence requirement. Retrieval is run for each sub-question; a downstream answer step can then combine the evidence while making sure all requested parts are addressed.

For example, “What are the Model X battery replacement steps, how long is its warranty, and where do I get repair support?” has at least three requirements:

1. Find Model X battery replacement instructions.
2. Find the duration and scope of the Model X warranty.
3. Find how to contact repair support for Model X.

These facts may live in different chunks. A single ranked list can return one highly relevant chunk while silently missing the other required facts. Decomposition makes those gaps visible.

### Decomposition is not query expansion

- **Query expansion** produces alternate phrasings of the *same* information need. Their results are usually merged to improve candidate coverage for one intent.
- **Query decomposition** creates *different but related* information needs, each of which must be answered to satisfy the original question.

For a compound request, paraphrasing the entire question several ways does not guarantee that each requested part is retrieved. Conversely, decomposing a simple question into several near-duplicate searches adds cost and can create inconsistent results without adding useful coverage.

### A useful decomposition preserves intent and relationships

A good sub-question is specific enough to retrieve its evidence, but it should not introduce facts the user did not ask for. Keep shared entities and constraints—such as product model, date, language, or tenant—in each relevant sub-question or apply them as structured filters. If a question has dependencies (“compare the warranty for each model”), record those dependencies rather than turning the request into unrelated searches.

## 2. A synthetic corpus with separate evidence

Each record is one chunk. The three target facts are deliberately stored in different chunks. Other passages mention similar terms but concern another model or a related topic, making it possible to see why entity constraints matter.

In [ ]:
corpus = [
    {'chunk_id': 'P01', 'topic': 'battery_steps', 'text': 'Model X battery replacement: turn the device off before removing the battery cover. Use the approved replacement battery.'},
    {'chunk_id': 'P02', 'topic': 'warranty', 'text': 'Model X warranty coverage lasts two years for manufacturing defects. Keep proof of purchase for a claim.'},
    {'chunk_id': 'P03', 'topic': 'repair_support', 'text': 'For Model X repair support, contact the service center and provide the device serial number.'},
    {'chunk_id': 'P04', 'topic': 'battery_steps', 'text': 'Model Y battery replacement: disconnect the charging cable before opening the battery compartment.'},
    {'chunk_id': 'P05', 'topic': 'warranty', 'text': 'Model Y includes a one-year limited warranty for manufacturing defects.'},
    {'chunk_id': 'P06', 'topic': 'general_support', 'text': 'For general account support, contact the help desk using the email address on your profile.'},
    {'chunk_id': 'P07', 'topic': 'charging', 'text': 'Charge Model X using the supplied adapter. Do not use a damaged cable.'},
    {'chunk_id': 'P08', 'topic': 'returns', 'text': 'Eligible products can be returned within thirty days of delivery.'},
    {'chunk_id': 'P09', 'topic': 'archived_warranty', 'text': 'Archived Model X warranty information is out of date and should not be used for current coverage.'},
]

original_question = (
    'What are the Model X battery replacement steps, how long is its warranty, '
    'and where do I get repair support?'
)

print('Original question:', original_question)
print('Chunks:', len(corpus))
for chunk in corpus[:3]:
    print(chunk['chunk_id'], chunk['topic'], '|', chunk['text'])

For the example, the relevant evidence is labeled as follows:

| Sub-question | Relevant chunk | Evidence needed |
| --- | --- | --- |
| Battery replacement steps | P01 | Model X-specific replacement instructions |
| Warranty duration and scope | P02 | Current Model X coverage duration and scope |
| Repair support contact | P03 | Model X repair support route and required information |

`P04` and `P05` are useful distractors: they discuss the same topics but refer to Model Y. `P09` is about Model X but explicitly archived. Correct retrieval must match both the topic and the requested entity/currentness.

## 3. Establish a whole-question baseline

We use a small lexical overlap retriever so each score is easy to explain. It counts distinct query terms shared with a chunk after a modest stop-word cleanup. This is a teaching baseline—not BM25, a dense model, or a recommendation for production. The lesson here is how a compound request can be searched and evaluated by parts.

In [ ]:
import re
from collections import defaultdict

STOP_WORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'do', 'for', 'how', 'i',
    'in', 'is', 'it', 'its', 'of', 'on', 'the', 'to', 'what', 'where',
    'with',
}

def tokenize(text):
    return set(re.findall(r'[a-z0-9]+', text.lower())) - STOP_WORDS

def score(query, text):
    return len(tokenize(query) & tokenize(text))

def retrieve(query, chunks=corpus, k=5):
    if k < 0:
        raise ValueError('k must be non-negative')
    scored = []
    for chunk in chunks:
        value = score(query, chunk['text'])
        if value > 0:
            scored.append({**chunk, 'score': value})
    return sorted(scored, key=lambda item: (-item['score'], item['chunk_id']))[:k]

whole_question_results = retrieve(original_question, k=5)
for rank, item in enumerate(whole_question_results, start=1):
    print(f"{rank}. {item['chunk_id']} score={item['score']} topic={item['topic']}")

The whole question contains terms for all three requirements. A lexical ranker can therefore give a single passage some points for matching only one part, while other parts are not represented in that top-K. Inspecting only the aggregate list may conceal this. Decomposition turns the implicit requirement—answer every requested part—into explicit retrieval tasks.

## 4. Decompose the question

For this notebook, the sub-questions are written by hand. This makes the intended meaning clear before we discuss automated decomposition. Each sub-question repeats `Model X`, a key entity constraint. In a larger system, mandatory metadata constraints should also be applied as filters rather than trusting generated query text alone.

In [ ]:
sub_questions = {
    'battery_steps': 'Model X battery replacement steps instructions',
    'warranty': 'Model X warranty duration years coverage defects',
    'repair_support': 'Model X repair support contact service center serial number',
}

for name, sub_question in sub_questions.items():
    print(f'{name:16s} {sub_question}')

Three sub-questions are appropriate here because the original request contains three independently answerable facts. Do not split every noun phrase into its own query: “Model X battery replacement steps” is one coherent need, not four. Over-decomposition can fragment context, multiply retrieval calls, and make it harder to combine the evidence correctly.

## 5. Retrieve separately for every sub-question

Each sub-question gets its own ranking. Keep the rankings separate at first so it is clear which chunk was retrieved for which requirement. This per-part view helps diagnose a missing answer even when the overall result union appears large.

In [ ]:
rankings = {
    name: retrieve(sub_question, k=4)
    for name, sub_question in sub_questions.items()
}

for name, results in rankings.items():
    print(f'\n{name}:')
    for rank, item in enumerate(results, start=1):
        print(f"  {rank}. {item['chunk_id']} score={item['score']} | {item['text']}")

For the carefully phrased sub-questions, the target chunks should rank first. Notice that the battery question may still return Model Y's passage because it shares many terms. An explicit model constraint or metadata filter can prevent this; a text score alone does not guarantee entity correctness.

### Relevance labels by sub-question

A single global relevant set is not enough to explain which part of the question is covered. We label relevant evidence separately for each sub-question. In real evaluations, use clear labeling guidance and assess whether chunks are sufficient evidence, not merely on-topic.

In [ ]:
relevance_by_part = {
    'battery_steps': {'P01'},
    'warranty': {'P02'},
    'repair_support': {'P03'},
}

assert set(relevance_by_part) == set(sub_questions)
for name, relevant_ids in relevance_by_part.items():
    print(name, '->', sorted(relevant_ids))

## 6. Measure coverage per sub-question

For each sub-question, define a hit at K as finding at least one relevant chunk in that sub-question's top K. **Sub-question coverage@K** is the fraction of sub-questions with a hit. This is different from ordinary chunk recall: it measures whether the system has evidence for every distinct requirement.

For this dataset, one relevant chunk is labeled per part, so per-part recall@K is either 0 or 1. In real tasks, a part may need multiple passages; use the full relevant set for that part and calculate recall accordingly.

In [ ]:
def part_hit_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    return any(item['chunk_id'] in relevant_ids for item in results[:k])

def subquestion_coverage_at_k(rankings, relevance, k):
    hits = {
        name: part_hit_at_k(rankings[name], relevance[name], k)
        for name in rankings
    }
    return sum(hits.values()) / len(hits), hits

for k in (1, 2, 3, 4):
    coverage, hits = subquestion_coverage_at_k(rankings, relevance_by_part, k)
    print(f'K={k}: coverage={coverage:.2f}; per-part hits={hits}')

If coverage is below 1, the system lacks judged evidence for at least one requested part. A downstream answer step should not silently invent that missing fact. Depending on the application, it may answer supported parts and explicitly say which part could not be verified, ask a clarifying question, or trigger another retrieval strategy.

## 7. Merge evidence without losing its meaning

A simple union of the top results for each sub-question is often a useful starting point. Deduplicate by stable chunk ID, but preserve a list of the sub-questions each chunk supports. A passage can support more than one part. Do not merge text into an untraceable blob: provenance helps a reviewer or generator connect claims to evidence.

In [ ]:
def merge_by_chunk_id(rankings, per_part_k=3):
    merged = {}
    for part_name, results in rankings.items():
        for rank, item in enumerate(results[:per_part_k], start=1):
            chunk_id = item['chunk_id']
            if chunk_id not in merged:
                merged[chunk_id] = {
                    **{key: value for key, value in item.items() if key != 'score'},
                    'part_hits': [],
                    'best_rank': rank,
                }
            merged[chunk_id]['part_hits'].append({'part': part_name, 'rank': rank})
            merged[chunk_id]['best_rank'] = min(merged[chunk_id]['best_rank'], rank)
    return sorted(merged.values(), key=lambda item: (item['best_rank'], item['chunk_id']))

merged_evidence = merge_by_chunk_id(rankings, per_part_k=3)
for item in merged_evidence:
    parts = ', '.join(hit['part'] for hit in item['part_hits'])
    print(f"{item['chunk_id']} supports [{parts}] | {item['text']}")

A merge can be ordered by per-part rank, a fused score, or a downstream reranker. There is no universally correct ordering: if every part is required, preserving at least one strong passage for each part may matter more than one global score. For large context budgets, allocate evidence per sub-question before filling remaining space with globally ranked candidates.

### Optional: fuse per-part rankings with RRF

RRF can produce one overall ordering across sub-question rankings. Here each sub-question is treated like one ranking source. That is convenient, but it can reward a chunk that is highly ranked for several similar sub-questions; it does not know whether those parts are redundant or whether every distinct requirement is covered. Keep the per-part coverage table alongside the fused list.

In [ ]:
def rrf_merge(rankings, c=60, per_part_k=5):
    if c <= 0:
        raise ValueError('c must be positive')
    scores = defaultdict(float)
    sources = defaultdict(list)
    records = {}
    for part, results in rankings.items():
        for rank, item in enumerate(results[:per_part_k], start=1):
            chunk_id = item['chunk_id']
            records[chunk_id] = {key: value for key, value in item.items() if key != 'score'}
            scores[chunk_id] += 1 / (c + rank)
            sources[chunk_id].append(part)
    fused = []
    for chunk_id, value in scores.items():
        fused.append({**records[chunk_id], 'rrf_score': value, 'retrieved_for': sources[chunk_id]})
    return sorted(fused, key=lambda item: (-item['rrf_score'], item['chunk_id']))

fused_evidence = rrf_merge(rankings, c=60, per_part_k=3)
for item in fused_evidence:
    print(item['chunk_id'], round(item['rrf_score'], 5), item['retrieved_for'])

## 8. Coverage chart: are all requested parts supported?

The first chart compares per-sub-question hit rates as K grows. Each bar is the percentage of the original question's sub-questions that have at least one relevant chunk in their own top K. A high aggregate chunk count can still hide a missing part; this chart keeps the parts separate.

In [ ]:
from html import escape
from xml.etree import ElementTree

try:
    from IPython.display import SVG, display
except ImportError:
    SVG = None
    display = None

def render_svg(svg_text):
    ElementTree.fromstring(svg_text)
    if SVG is not None and display is not None:
        display(SVG(svg_text))
    else:
        print('SVG is valid. Run in Jupyter/IPython to display inline.')

def coverage_chart_svg(k_values, coverage_values):
    width, height = 760, 360
    left, right, top, bottom = 80, 35, 55, 65
    plot_w, plot_h = width - left - right, height - top - bottom
    bar_w = plot_w / len(k_values) * 0.55
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="30" font-family="Arial" font-size="19" font-weight="bold" fill="#111827">Sub-question coverage@K</text>',
    ]
    for tick in (0, 0.25, 0.5, 0.75, 1.0):
        y = top + plot_h * (1 - tick)
        parts.append(f'<line x1="{left}" y1="{y:.1f}" x2="{width-right}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        parts.append(f'<text x="{left-10}" y="{y+4:.1f}" text-anchor="end" font-family="Arial" font-size="11" fill="#6b7280">{tick:.0%}</text>')
    for index, (k, value) in enumerate(zip(k_values, coverage_values)):
        center = left + (index + 0.5) * plot_w / len(k_values)
        bar_h = plot_h * value
        y = top + plot_h - bar_h
        parts.append(f'<rect x="{center-bar_w/2:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" rx="4" fill="#0f766e"/>')
        parts.append(f'<text x="{center:.1f}" y="{y-8:.1f}" text-anchor="middle" font-family="Arial" font-size="12" fill="#111827">{value:.0%}</text>')
        parts.append(f'<text x="{center:.1f}" y="{height-bottom+24}" text-anchor="middle" font-family="Arial" font-size="12" fill="#374151">{k}</text>')
    parts.append(f'<text x="{width/2}" y="{height-15}" text-anchor="middle" font-family="Arial" font-size="12" fill="#374151">Per-sub-question K</text>')
    parts.append('</svg>')
    return ''.join(parts)

coverage_ks = [1, 2, 3, 4]
coverage_values = [subquestion_coverage_at_k(rankings, relevance_by_part, k)[0] for k in coverage_ks]
render_svg(coverage_chart_svg(coverage_ks, coverage_values))

In this small dataset, coverage should be complete at K=1 because each manually written sub-question matches its target passage. That is a useful demonstration, not evidence that decomposition always succeeds. Add ambiguous queries, missing evidence, wrong entities, and less carefully phrased questions to create a meaningful evaluation set.

## 9. Evidence matrix chart

The second chart maps chunks to sub-questions. Green cells indicate a labeled relevant chunk retrieved in that sub-question's top K; gray cells are retrieved but not labeled relevant; blank cells were not returned. This makes false matches, missing evidence, and overlap across parts visible.

In [ ]:
def evidence_matrix_svg(rankings, relevance, k=3):
    parts_names = list(rankings)
    chunk_ids = sorted({item['chunk_id'] for results in rankings.values() for item in results[:k]})
    label_w, cell_w, top, row_h = 155, 94, 88, 42
    width = label_w + len(parts_names) * cell_w + 30
    height = top + len(chunk_ids) * row_h + 30
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        '<text x="12" y="30" font-family="Arial" font-size="18" font-weight="bold" fill="#111827">Retrieved evidence by sub-question (top K)</text>',
    ]
    for col, part in enumerate(parts_names):
        x = label_w + col * cell_w + cell_w / 2
        label = part.replace('_', ' ')
        parts.append(f'<text x="{x:.1f}" y="68" text-anchor="middle" font-family="Arial" font-size="11" fill="#374151">{escape(label)}</text>')
    for row, chunk_id in enumerate(chunk_ids):
        y = top + row * row_h
        parts.append(f'<text x="{label_w-12}" y="{y+23}" text-anchor="end" font-family="Arial" font-size="12" fill="#374151">{escape(chunk_id)}</text>')
        for col, part in enumerate(parts_names):
            x = label_w + col * cell_w
            results = rankings[part][:k]
            present = any(item['chunk_id'] == chunk_id for item in results)
            relevant = chunk_id in relevance[part]
            fill = '#bbf7d0' if present and relevant else ('#e5e7eb' if present else '#ffffff')
            text = '✓' if present and relevant else ('•' if present else '')
            parts.append(f'<rect x="{x+3}" y="{y+3}" width="{cell_w-6}" height="{row_h-6}" rx="4" fill="{fill}" stroke="#d1d5db"/>')
            parts.append(f'<text x="{x+cell_w/2}" y="{y+24}" text-anchor="middle" font-family="Arial" font-size="16" fill="#166534">{text}</text>')
    parts.append(f'<text x="{label_w}" y="{height-8}" font-family="Arial" font-size="10" fill="#166534">✓ relevant</text>')
    parts.append(f'<text x="{label_w+92}" y="{height-8}" font-family="Arial" font-size="10" fill="#4b5563">• retrieved, not labeled relevant</text>')
    parts.append('</svg>')
    return ''.join(parts)

render_svg(evidence_matrix_svg(rankings, relevance_by_part, k=3))

A gray mark is not automatically a harmful false positive: relevance labels may be incomplete, or the chunk may provide useful context. The chart is a prompt for inspection. Review the text and the task-specific judgment before deciding whether to change query wording, filters, chunking, or ranking.

## 10. What should happen if one part has no evidence?

A retrieval system should not treat “some evidence found” as “the whole compound question is answered.” If a sub-question has no relevant candidate, possible next steps include:

- Try another retrieval method or a constrained query rewrite for that part.
- Ask a clarifying question if the sub-question is ambiguous.
- Tell the user which part could not be verified from the available sources.
- Search another authorized corpus if one exists.
- Stop and avoid an unsupported answer if the evidence is absent.

Do not broaden access filters or drop mandatory constraints merely to fill a missing part. Keep the distinction between no matching result, no relevant result, and no source evidence.

## 11. Automating decomposition carefully

An LLM can propose sub-questions, but the decomposition itself must be validated. A practical process is:

1. Ask for a minimal list of distinct facts required to answer the original question.
2. Require each sub-question to retain relevant entities, dates, units, qualifiers, and negation.
3. Check that every sub-question is grounded in the original request and that the set covers the whole request.
4. Reject duplicate or over-fragmented sub-questions.
5. Apply structured filters independently from generated wording.
6. Retrieve per part, preserve provenance, and measure coverage.

For high-impact applications, decomposition should be observable and testable. Keep test cases for compound questions, conditional questions, comparisons, lists, and no-answer cases.

## 12. Common failure modes

| Failure | Example | Mitigation |
| --- | --- | --- |
| Under-decomposition | One sub-question covers only the battery steps, dropping warranty and support. | Check coverage of the original request's requested facts. |
| Over-decomposition | Separate searches for “Model,” “X,” “battery,” and “steps.” | Decompose by independently answerable evidence requirements. |
| Entity loss | A rewrite asks about a product but drops “Model X.” | Repeat entity constraints or apply trusted metadata filters. |
| Query drift | Warranty question changes from current coverage to archived coverage. | Preserve qualifiers and inspect generated sub-questions. |
| Evidence mismatch | Model Y results retrieved for Model X. | Use entity-aware retrieval and explicit filters; inspect text. |
| Missing answer treated as complete | Two of three parts have evidence, but the final response answers all three. | Track per-part coverage and require evidence for each claim. |
| Untraceable merge | Passage list is deduplicated but support relationships are lost. | Preserve chunk IDs, sub-question IDs, and source provenance. |
| Contradictory sources | One part retrieves current and archived guidance. | Include date/version metadata, source precedence, and conflict handling. |

Decomposition improves structure and observability; it does not guarantee correct retrieval or answer synthesis.

## 13. Exercises

1. Add a fourth part to the original question: “Does the device support fast charging?” Add a chunk and relevance label, then measure coverage again.
2. Make a compound question with two dependent parts, such as “Which model has the longer warranty, and what is that model's battery procedure?” What relationships must the decomposition preserve?
3. Remove the Model X phrase from one sub-question. Does Model Y become a stronger false match? Try a metadata filter to enforce the model.
4. Simulate missing evidence for warranty by temporarily removing P02. What should the coverage metric show, and how should a downstream answer communicate the gap?
5. Change K and compare per-part coverage with total unique candidate count. Which setting gives a useful balance for this small corpus?
6. Write an intentionally over-decomposed set of sub-questions. How many redundant calls does it produce, and does coverage improve?
7. Create a held-out set of at least five compound questions, label evidence per part, and compare whole-question retrieval with decomposition.
8. If an LLM generates the sub-questions, create checks for entity retention, negation, completeness, and duplicate parts.

### Optional extension: invariants

The checks below verify that every original request part has a label, the merge deduplicates chunks, every returned chunk has provenance, and invalid K/RRF inputs are rejected.

In [ ]:
assert set(sub_questions) == set(relevance_by_part)
assert len({item['chunk_id'] for item in merged_evidence}) == len(merged_evidence)
assert all(item['part_hits'] for item in merged_evidence)
assert subquestion_coverage_at_k(rankings, relevance_by_part, 1)[0] == 1.0

try:
    retrieve('test', k=-1)
except ValueError:
    print('Negative K rejected as expected.')
else:
    raise AssertionError('Negative K should raise ValueError')

try:
    rrf_merge(rankings, c=0)
except ValueError:
    print('Non-positive RRF constant rejected as expected.')
else:
    raise AssertionError('Non-positive c should raise ValueError')

print('Coverage, deduplication, provenance, and input checks passed.')

## Summary

Query decomposition turns a compound question into a small set of distinct, answerable retrieval tasks. Search each part separately, preserve its constraints, track evidence provenance, and measure whether every part is covered. Merge duplicate chunks by stable ID without losing which sub-question they support. Coverage by sub-question reveals gaps that a global result list can hide.

Decomposition is not paraphrase expansion: it creates separate requirements rather than alternate wordings of one requirement. It can add calls, drift, wrong-entity matches, or unsupported assumptions. Validate the decomposition, evaluate coverage on judged compound questions, and clearly report missing evidence instead of silently filling gaps. The next notebook moves from retrieving chunks to **parent-document retrieval**, where focused matches can be expanded into broader context.